# EEG_08b — GNN Subject-Specific Classification + Subject Ranking

Trains a GCN per subject (leave-one-session-out split) and ranks subjects
by test balanced accuracy.

| Section | Description |
|---------|-------------|
| §1 | Imports & Config |
| §2 | Dataset + session split |
| §3 | GCN model |
| §4 | Train / eval functions |
| §5 | Per-subject loop |
| §6 | Ranking + plot |

## 1 — Imports & Config

In [ ]:
import json
import logging
import random
import re
import time
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score
from torch.utils.data import Dataset as TorchDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg08b')

project_root = next(
    (p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / '.git').exists()),
    Path.cwd(),
)
(project_root / 'checkpoints').mkdir(exist_ok=True)
(project_root / 'figures').mkdir(exist_ok=True)
log.info(f'project_root: {project_root}')

CONFIG = {
    'data_root':      str(project_root / 'data' / 'graphs_abs_pcc'),
    'n_classes':      4,
    'batch_size':     32,
    'epochs':         80,
    'lr':             1e-3,
    'early_stopping': 15,
    'seed':           42,
    'device':         'cuda' if torch.cuda.is_available() else 'cpu',
    'hidden_dim':     64,
    'dropout':        0.3,
    'wandb_project':  'miralis-imagined-speech',
    'wandb_entity':   'uras-daniele22-politecnico-di-milano',
    'wandb_group':    'eeg08b_subject_specific',
}

# Normalizzazione per-trial (Bomatter 2024)
# True  = z-score per-canale per trial (rimuove variabilita inter-sessione)
# False = segnale grezzo dal .pt
USE_INSTANCE_NORM = True
log.info(f'USE_INSTANCE_NORM={USE_INSTANCE_NORM}')

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
n_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
log.info(f"device={CONFIG['device']}  seed={CONFIG['seed']}")


## 2 — Dataset & Session Split

**Split strategy**: leave-one-session-out.
- Test  → ultima sessione (S005)
- Val   → penultima (S004)
- Train → tutte le altre (S001-S003)

Fallback a random 70/15/15 se soggetto ha < 3 sessioni.

In [ ]:
_PAT       = re.compile(r'^P(\d+)_S(\d+)$')
_data_root = Path(CONFIG['data_root'])
assert _data_root.exists(), f'data_root not found: {_data_root}'

# label mapping
_raw_map = json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()
)
LABEL2CLUSTER = torch.zeros(110, dtype=torch.long)
for ws, cid in _raw_map.items():
    LABEL2CLUSTER[int(ws)] = int(cid)

# raccolta path per soggetto -> sessione
subj_sess_paths = defaultdict(lambda: defaultdict(list))
for p in sorted(_data_root.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess_paths[int(m.group(1))][int(m.group(2))].append(p)

ALL_SUBJ_IDS = sorted(subj_sess_paths.keys())
log.info(f'Soggetti: {len(ALL_SUBJ_IDS)}')

def session_split(subj_id):
    sess = subj_sess_paths[subj_id]
    sids = sorted(sess.keys())
    if len(sids) >= 3:
        train_paths = [p for s in sids[:-2] for p in sess[s]]
        val_paths   = list(sess[sids[-2]])
        test_paths  = list(sess[sids[-1]])
    else:
        all_p = [p for s in sids for p in sess[s]]
        rng   = random.Random(CONFIG['seed'])
        rng.shuffle(all_p)
        n = len(all_p)
        n_tr = int(n * 0.70); n_va = int(n * 0.15)
        train_paths = all_p[:n_tr]
        val_paths   = all_p[n_tr:n_tr+n_va]
        test_paths  = all_p[n_tr+n_va:]
    return train_paths, val_paths, test_paths


class EEGGraphDataset(TorchDataset):
    def __init__(self, pt_paths):
        self.paths = list(pt_paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        d  = torch.load(self.paths[idx], weights_only=False)
        ea = d['edge_attr']
        if not isinstance(ea, torch.Tensor):
            ea = torch.tensor(np.asarray(ea), dtype=torch.float32)
        ea = ea.float()
        ew = ea if ea.dim() == 1 else ea[:, 0]
        y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
        x = d['x'].float()
        if USE_INSTANCE_NORM:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return Data(x=x, edge_index=d['edge_index'].long(),
                    edge_weight=ew, y=LABEL2CLUSTER[y_word])


def build_loaders(tr, va, te):
    bs = CONFIG['batch_size']
    return (
        PyGDataLoader(EEGGraphDataset(tr), batch_size=bs, shuffle=True),
        PyGDataLoader(EEGGraphDataset(va), batch_size=bs, shuffle=False),
        PyGDataLoader(EEGGraphDataset(te), batch_size=bs, shuffle=False),
    )


def compute_class_weights(paths):
    labels = []
    for p in paths:
        d = torch.load(p, weights_only=False)
        y = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
        labels.append(int(LABEL2CLUSTER[y]))
    counts = np.bincount(labels, minlength=CONFIG['n_classes']).astype(float)
    counts = np.where(counts == 0, 1, counts)
    return torch.tensor(len(labels) / (CONFIG['n_classes'] * counts), dtype=torch.float32)


_s = torch.load(next(_data_root.rglob('trial_*.pt')), weights_only=False)
IN_CHANNELS = int(_s['x'].shape[1])
log.info(f'IN_CHANNELS={IN_CHANNELS}')


## 3 — GCN Model

In [ ]:
def _mlp_head(in_dim, out_dim, dropout):
    return nn.Sequential(
        nn.Linear(in_dim, in_dim // 2), nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(in_dim // 2, out_dim),
    )


class GCNModel(nn.Module):
    def __init__(self, in_channels, hidden_dim, n_classes, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim,  hidden_dim)
        self.conv3 = GCNConv(hidden_dim,  hidden_dim)
        self.drop  = nn.Dropout(dropout)
        self.head  = _mlp_head(hidden_dim, n_classes, dropout)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        ew = getattr(data, 'edge_weight', None)
        x = F.relu(self.conv1(x, ei, ew)); x = self.drop(x)
        x = F.relu(self.conv2(x, ei, ew)); x = self.drop(x)
        x = F.relu(self.conv3(x, ei, ew))
        return self.head(global_mean_pool(x, data.batch))


log.info('GCN OK')


## 4 — Train & Evaluate

In [ ]:
def train_one_subject(subj_id, train_loader, val_loader, class_weights):
    device = torch.device(CONFIG['device'])
    model  = GCNModel(IN_CHANNELS, CONFIG['hidden_dim'],
                      CONFIG['n_classes'], CONFIG['dropout']).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    crit   = nn.CrossEntropyLoss(weight=class_weights.to(device))
    ckpt   = project_root / 'checkpoints' / f'eeg08b_P{subj_id:03d}.pt'

    run = wandb.init(
        project=CONFIG['wandb_project'], entity=CONFIG['wandb_entity'],
        group=CONFIG['wandb_group'],
        name=f"P{subj_id:03d}_GCN{'_norm' if USE_INSTANCE_NORM else ''}",
        config={**CONFIG, 'subject_id': subj_id,
                'use_instance_norm': USE_INSTANCE_NORM},
        reinit='finish_previous',
    )

    best_val_bacc, patience = -1.0, 0
    for epoch in range(CONFIG['epochs']):
        model.train()
        tr_preds, tr_labels = [], []
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            loss = nn.CrossEntropyLoss(weight=class_weights.to(device))(
                model(batch), batch.y)
            loss.backward(); opt.step()
            tr_preds.extend(model(batch).detach().argmax(1).cpu().numpy())
            tr_labels.extend(batch.y.cpu().numpy())

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                val_preds.extend(model(batch).argmax(1).cpu().numpy())
                val_labels.extend(batch.y.cpu().numpy())

        vba = balanced_accuracy_score(val_labels, val_preds)
        wandb.log({'train/bacc': balanced_accuracy_score(tr_labels, tr_preds),
                   'val/bacc': vba, 'epoch': epoch})

        if vba > best_val_bacc:
            best_val_bacc = vba; patience = 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience += 1
        if patience >= CONFIG['early_stopping']:
            break

    model.load_state_dict(torch.load(ckpt, weights_only=True))
    run.summary['best_val_bacc'] = best_val_bacc
    run.finish()
    return model


def evaluate(model, loader):
    device = torch.device(CONFIG['device'])
    model.to(device).eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds.extend(model(batch).argmax(1).cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    preds, labels = np.array(preds), np.array(labels)
    return {
        'test_bacc': balanced_accuracy_score(labels, preds),
        'macro_f1':  f1_score(labels, preds, average='macro', zero_division=0),
        'cm':        confusion_matrix(labels, preds, labels=list(range(CONFIG['n_classes']))),
    }


## 5 — Per-Subject Training Loop

In [ ]:
wandb.login()

SUBJECT_RESULTS = {}  # subj_id -> metrics dict

for subj_id in tqdm(ALL_SUBJ_IDS, desc='Subjects'):
    tr_paths, va_paths, te_paths = session_split(subj_id)
    if not te_paths:
        log.warning(f'P{subj_id:03d}: nessun trial di test, skip')
        continue

    tr_loader, va_loader, te_loader = build_loaders(tr_paths, va_paths, te_paths)
    cw = compute_class_weights(tr_paths)

    t0    = time.time()
    model = train_one_subject(subj_id, tr_loader, va_loader, cw)
    res   = evaluate(model, te_loader)
    res['train_time_s'] = round(time.time() - t0, 1)
    res['n_train'] = len(tr_paths)
    res['n_val']   = len(va_paths)
    res['n_test']  = len(te_paths)

    SUBJECT_RESULTS[subj_id] = res
    log.info(f"P{subj_id:03d}  test_bacc={res['test_bacc']:.4f}  "
             f"f1={res['macro_f1']:.4f}  t={res['train_time_s']}s")

log.info(f'Completati {len(SUBJECT_RESULTS)}/{len(ALL_SUBJ_IDS)} soggetti')


## 6 — Subject Ranking

In [ ]:
# tabella ranking
rows = []
for sid, res in SUBJECT_RESULTS.items():
    rows.append({
        'Subject':   f'P{sid:03d}',
        'Test bAcc': round(res['test_bacc'], 4),
        'Macro F1':  round(res['macro_f1'],  4),
        'N Train':   res['n_train'],
        'N Test':    res['n_test'],
        'Time (s)':  res['train_time_s'],
    })

df_rank = (pd.DataFrame(rows)
             .sort_values('Test bAcc', ascending=False)
             .reset_index(drop=True))
df_rank.index = df_rank.index + 1
df_rank.index.name = 'Rank'

chance = 1 / CONFIG['n_classes']
print(df_rank.to_string())
print(f'\nChance level : {chance:.2%}')
print(f'Mean test bAcc: {df_rank["Test bAcc"].mean():.4f} +/- {df_rank["Test bAcc"].std():.4f}')
print(f'Best:  {df_rank.iloc[0]["Subject"]}  {df_rank.iloc[0]["Test bAcc"]:.4f}')
print(f'Worst: {df_rank.iloc[-1]["Subject"]}  {df_rank.iloc[-1]["Test bAcc"]:.4f}')

# bar chart
fig, ax = plt.subplots(figsize=(18, 5))
colors = ['#2ecc71' if v > chance + 0.05 else '#e74c3c'
          for v in df_rank['Test bAcc']]
ax.bar(df_rank['Subject'], df_rank['Test bAcc'], color=colors, edgecolor='none')
ax.axhline(chance, color='black', lw=1.5, linestyle='--',
           label=f'Chance ({chance:.0%})')
ax.set_xlabel('Subject (ranked best to worst)', fontsize=11)
ax.set_ylabel('Balanced Accuracy', fontsize=11)
ax.set_title('EEG_08b — Subject-Specific GCN | Test Balanced Accuracy',
             fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.legend()
plt.tight_layout()
fig.savefig(project_root / 'figures' / 'eeg08b_subject_ranking.png',
            dpi=150, bbox_inches='tight')
plt.show()

# salva CSV
out_csv = project_root / 'figures' / 'eeg08b_subject_ranking.csv'
df_rank.to_csv(out_csv)
print(f'Salvato: {out_csv}')

# ── confusion matrix top-5 soggetti ─────────────────────────────────────
LABEL_NAMES = ['CONCR', 'AZIONE', 'STATO', 'ASTRATTO']
top5 = df_rank.head(5)['Subject'].tolist()
top5_ids = [int(s[1:]) for s in top5]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Confusion Matrix — Top-5 Subjects (Subject-Specific GCN)',
             fontsize=13, fontweight='bold')

for ax, sid, sname in zip(axes, top5_ids, top5):
    res = SUBJECT_RESULTS[sid]
    cm  = res['cm'].astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels(LABEL_NAMES, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(LABEL_NAMES, fontsize=8)
    ax.set_title(f"{sname}\nbAcc={res['test_bacc']:.3f}  F1={res['macro_f1']:.3f}",
                 fontsize=9, fontweight='bold')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    fontsize=8, color='white' if cm_norm[i,j] > 0.5 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
fig.savefig(project_root / 'figures' / 'eeg08b_top5_confusion.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix top-5 salvata.')
